Bert

In [ ]:
import pandas as pd

# Load dataset
data = pd.read_csv('Stress.csv')

# Preprocess text data (Convert to lowercase)
data['text'] = data['text'].astype(str).apply(lambda x: x.lower())

# Check class distribution
print("Class Distribution Before Balancing:")
print(data['label'].value_counts())  # Check the number of samples per class


Class Distribution Before Balancing:
label
0    1350
2    1011
1     477
Name: count, dtype: int64


In [ ]:
# Handle class imbalance (Oversampling the minority classes)
class_counts = data['label'].value_counts()
max_count = class_counts.max()

balanced_data = []
for label in data['label'].unique():
    subset = data[data['label'] == label]
    balanced_data.append(subset.sample(max_count, replace=True))  # Oversample minority classes

data_balanced = pd.concat(balanced_data)

# Verify class distribution after balancing
print("Class Distribution After Balancing:")
print(data_balanced['label'].value_counts())

# Extract text and labels again
X = data_balanced['text']
y = data_balanced['label']


Class Distribution After Balancing:
label
2    1350
0    1350
1    1350
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from transformers import BertTokenizer

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the input text
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
import torch

# Define dataset class
class StressDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values)  # Convert labels to tensor

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Create dataset objects for training and testing
train_dataset = StressDataset(train_encodings, y_train)
test_dataset = StressDataset(test_encodings, y_test)


In [ ]:
from transformers import BertForSequenceClassification

# Load BERT model for sequence classification with 3 output labels
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',  # Directory to save model outputs
    num_train_epochs=20,  # Train for 20 epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    evaluation_strategy="epoch",  # Evaluate at every epoch
    save_strategy="epoch",  # Save model after every epoch
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Trainer

# Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: peeyushk-aids22 (peeyushk-aids22-salesforce) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,0.649667
2,0.873200,0.488540
3,0.462300,0.591661
4,0.205700,0.696838
5,0.098500,0.640675
6,0.098500,0.706619
7,0.042000,0.727668
8,0.014400,0.871803
9,0.019200,1.024759
10,0.011900,0.740704


TrainOutput(global_step=8100, training_loss=0.109810414585992, metrics={'train_runtime': 2455.7955, 'train_samples_per_second': 26.387, 'train_steps_per_second': 3.298, 'total_flos': 4262437367193600.0, 'train_loss': 0.109810414585992, 'epoch': 20.0})

In [ ]:
from sklearn.metrics import classification_report

# Evaluate the model on test data
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, pred_labels, target_names=['Low', 'Moderate', 'High']))



Classification Report:
              precision    recall  f1-score   support

         Low       0.96      0.88      0.92       270
    Moderate       0.85      0.91      0.88       270
        High       0.89      0.90      0.89       270

    accuracy                           0.90       810
   macro avg       0.90      0.90      0.90       810
weighted avg       0.90      0.90      0.90       810



In [ ]:
def classify_stress_level(user_input):
    inputs = tokenizer(user_input, return_tensors='pt', truncation=True, padding=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax().item()
    stress_levels = ['Low', 'Moderate', 'High']
    return stress_levels[predicted_class]


In [ ]:
def classify_stress_level(user_input):
    inputs = tokenizer(user_input, return_tensors='pt', truncation=True, padding=True, max_length=128)

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move tensors to the device where the model is located

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax().item()
    stress_levels = ['Low', 'Moderate', 'High']
    return stress_levels[predicted_class]

In [ ]:
while True:
    user_input = input("Enter a sentence to classify stress level (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    stress_level = classify_stress_level(user_input)
    print(f"Predicted Stress Level: {stress_level}")


Enter a sentence to classify stress level (or type 'exit' to quit): I feel relaxed and happy today.
Predicted Stress Level: Low
Enter a sentence to classify stress level (or type 'exit' to quit): I have so many deadlines coming up, it's getting stressful.
Predicted Stress Level: High
Enter a sentence to classify stress level (or type 'exit' to quit): The workload is increasing, but I'm managing somehow
Predicted Stress Level: Low
Enter a sentence to classify stress level (or type 'exit' to quit):  can't handle this anymore, everything is falling apart!
Predicted Stress Level: High
Enter a sentence to classify stress level (or type 'exit' to quit): I don't know if I can finish everything on time.
Predicted Stress Level: High
Enter a sentence to classify stress level (or type 'exit' to quit): I am happy
Predicted Stress Level: Low
Enter a sentence to classify stress level (or type 'exit' to quit): No Stress
Predicted Stress Level: Low
Enter a sentence to classify stress level (or type 'e